# Transfer Learning con MobileNetV3Small
Notebook preparado para clasificación binaria de imágenes de **220×220×3**.

Flujo:
1. Cargar `X_train.npy`, `y_train.npy`, `X_val.npy`, `y_val.npy`.
2. Construir MobileNetV3Small con pesos de ImageNet.
3. Entrenar primero con el backbone congelado.
4. Realizar fine-tuning de las últimas capas.
5. Graficar curvas y evaluar con matriz de confusión y métricas.


In [1]:
# Herramienta de Google Colab para conectar el notebook con Google Drive
from google.colab import drive
# Monta (conecta) Google Drive en este entorno para poder leer y guardar archivos ahí
drive.mount('/content/MyDrive')

ValueError: mount failed

In [2]:
import os
# Cambia la carpeta de trabajo actual a la carpeta del proyecto dentro de Drive
os.chdir('/content/MyDrive/MyDrive/Proyecto_glaucoma')

In [3]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [4]:
!nvidia-smi

In [5]:
# NumPy: librería para trabajar con arreglos numéricos (matrices) de forma eficiente
import numpy as np
# Matplotlib: librería para crear gráficos y mostrar imágenes
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


## 1. Cargar datos
Modificar `BASE_PATH` si es necesario

In [6]:
BASE_PATH = 'data_Alltrain_Allval_240test'

# Carga un arreglo NumPy previamente guardado en disco (archivo .npy)
X_train = np.load(f'{BASE_PATH}/train/X_train.npy')
y_train = np.load(f'{BASE_PATH}/train/y_train.npy')

X_val = np.load(f'{BASE_PATH}/val/X_val.npy')
y_val = np.load(f'{BASE_PATH}/val/y_val.npy')

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape)
print('X_val:', X_val.shape, X_val.dtype)
print('y_val:', y_val.shape)

# Cuenta cuántos elementos únicos hay (aquí, cuántas imágenes hay de cada clase)
print('Clases train:', np.unique(y_train, return_counts=True))
print('Clases val:', np.unique(y_val, return_counts=True))


## 2. MobileNetV3Small
`MobileNetV3Small` de Keras incluye por defecto el preprocesamiento necesario cuando `include_preprocessing=True`.
Por eso conservamos las imágenes en el rango **0–255** y no hacemos `X / 255` antes del modelo.


In [7]:
# Carga la arquitectura MobileNetV3Small, pre-entrenada en el dataset ImageNet (millones de imágenes genéricas)
base_model = MobileNetV3Small(
    input_shape=(220, 220, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True
)

# Primera etapa: congelamos MobileNet
# Congela estas capas: no se van a modificar durante el entrenamiento, para conservar lo que ya aprendieron
base_model.trainable = False

inputs = tf.keras.Input(shape=(220, 220, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
# Apaga aleatoriamente un porcentaje de neuronas durante el entrenamiento, para evitar que el modelo memorice en vez de aprender (sobreajuste)
x = Dropout(0.30)(x)
# Capa 'densa' o totalmente conectada: cada neurona recibe todas las salidas de la capa anterior
x = Dense(128, activation='relu')(x)
x = Dropout(0.30)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs, name='MobileNetV3Small_transfer')

model.summary()


## 3. Compilar

In [8]:
# Configura el modelo antes de entrenar: qué función de pérdida usar, qué optimizador, y qué métricas medir
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)


## 4. Callbacks

In [9]:
callbacks = [
    # Callback que detiene el entrenamiento automáticamente si el modelo deja de mejorar, ahorrando tiempo y evitando sobreajuste
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Callback que reduce automáticamente la tasa de aprendizaje cuando el modelo deja de progresar
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # Callback que guarda automáticamente la mejor versión del modelo durante el entrenamiento
    ModelCheckpoint(
        'MobileNetV3Small_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]


## 5. Etapa 1 — Feature extraction

In [10]:
# Entrena el modelo: le muestra los datos de entrenamiento repetidamente y ajusta sus pesos internos para reducir el error
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


## 6. Curvas de entrenamiento

In [11]:
def graficar_historial(history, titulo='MobileNetV3Small'):
    # Crea una nueva figura (lienzo) donde se dibujará el gráfico
    plt.figure(figsize=(8,5))
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.xlabel('Época')
    plt.ylabel('Accuracy')
    # Agrega un título al gráfico o imagen mostrada
    plt.title(f'{titulo} - Accuracy')
    plt.legend()
    plt.grid()
    # Renderiza (dibuja) el gráfico en pantalla
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.title(f'{titulo} - Loss')
    plt.legend()
    plt.grid()
    plt.show()

graficar_historial(history)


## 7. Etapa 2 — Fine-tuning
Descongelamos únicamente las últimas capas de MobileNet y usamos un *learning rate* mucho menor.


In [12]:
# Descongela estas capas para permitir el fine-tuning (ajuste fino) sobre nuestros datos
base_model.trainable = True

# Congelar todas excepto las últimas 30 capas
for layer in base_model.layers[:-30]:
    # Congela estas capas: no se van a modificar durante el entrenamiento, para conservar lo que ya aprendieron
    layer.trainable = False

print('Capas totales del backbone:', len(base_model.layers))
print('Capas entrenables:', sum(layer.trainable for layer in base_model.layers))

# Configura el modelo antes de entrenar: qué función de pérdida usar, qué optimizador, y qué métricas medir
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)


In [13]:
callbacks_ft = [
    # Callback que detiene el entrenamiento automáticamente si el modelo deja de mejorar, ahorrando tiempo y evitando sobreajuste
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Callback que reduce automáticamente la tasa de aprendizaje cuando el modelo deja de progresar
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # Callback que guarda automáticamente la mejor versión del modelo durante el entrenamiento
    ModelCheckpoint(
        'MobileNetV3Small_finetuned_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Entrena el modelo: le muestra los datos de entrenamiento repetidamente y ajusta sus pesos internos para reducir el error
history_ft = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks_ft,
    verbose=1
)


## 8. Evaluación final con el conjunto de TEST

El conjunto de validación (`X_val`, `y_val`) se utiliza durante el entrenamiento para seleccionar y ajustar el modelo.

Las métricas finales deben calcularse con un conjunto **independiente**, que no haya participado ni en el entrenamiento ni en el fine-tuning. Por eso, a partir de esta sección se utilizan exclusivamente `X_test` y `y_test`.


In [ ]:
from numpy import load

# Cargar datos reservados exclusivamente para TEST
X_test = load('data_Alltrain_Allval_240test/test/X_test.npy')
y_test = load('data_Alltrain_Allval_240test/test/y_test.npy')

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)
# Cuenta cuántos elementos únicos hay (aquí, cuántas imágenes hay de cada clase)
print('Clases en test:', np.unique(y_test, return_counts=True))


### 8.1 Métricas finales sobre TEST

Se evalúa el modelo entrenado sobre datos que nunca fueron utilizados para actualizar sus pesos.


In [ ]:
# Evalúa el modelo ya entrenado sobre un conjunto de datos, devolviendo la pérdida y las métricas configuradas
resultados_test = model.evaluate(
    X_test,
    y_test,
    batch_size=32,
    verbose=1,
    return_dict=True
)

print('\nRESULTADOS FINALES EN TEST')
print('-' * 40)

for nombre, valor in resultados_test.items():
    print(f'{nombre:12s}: {valor:.4f}')


## 9. Predicciones finales sobre TEST

Para la clasificación binaria, MobileNetV3Small produce una probabilidad entre 0 y 1. Se utiliza un umbral de **0.5** para convertir esa probabilidad en la clase predicha.


In [ ]:
# Probabilidad predicha para cada imagen del conjunto TEST
# Usa el modelo entrenado para generar predicciones sobre nuevas imágenes
y_prob = model.predict(
    X_test,
    batch_size=32,
    verbose=1
).ravel()

# Conversión de probabilidad a clase
y_pred = (y_prob >= 0.5).astype(np.int32)

print('Número de predicciones:', len(y_pred))
print('Primeras probabilidades:', y_prob[:10])
print('Primeras clases predichas:', y_pred[:10])


## 10. Reporte de clasificación sobre TEST

Se calculan **Precision, Recall y F1-score** para cada clase, además de los promedios globales.


In [ ]:
from sklearn.metrics import classification_report

print(
    # Genera un reporte con Precision, Recall y F1-score para cada clase
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)


## 11. Matriz de confusión sobre TEST

La matriz de confusión se genera utilizando **únicamente `y_test` y las predicciones de `X_test`**.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Matplotlib: librería para crear gráficos y mostrar imágenes
import matplotlib.pyplot as plt

# Calcula la matriz de confusión: compara las predicciones del modelo contra las etiquetas reales
cm = confusion_matrix(y_test, y_pred)

print('Matriz de confusión:')
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Clase 0', 'Clase 1']
)

disp.plot(values_format='d')
# Agrega un título al gráfico o imagen mostrada
plt.title('MobileNetV3Small - Matriz de confusión (TEST)')
# Renderiza (dibuja) el gráfico en pantalla
plt.show()


## 12. Curva ROC y AUC sobre TEST

La curva ROC utiliza directamente las probabilidades producidas por el modelo, sin aplicar previamente el umbral de 0.5.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Calcula los puntos de la curva ROC (relación entre verdaderos y falsos positivos a distintos umbrales)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
# Calcula el AUC (área bajo la curva ROC): una métrica de 0 a 1 de qué tan bien separa el modelo las dos clases
auc_test = roc_auc_score(y_test, y_prob)

# Crea una nueva figura (lienzo) donde se dibujará el gráfico
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'MobileNetV3Small (AUC = {auc_test:.4f})')
plt.plot([0, 1], [0, 1], '--', label='Clasificador aleatorio')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
# Agrega un título al gráfico o imagen mostrada
plt.title('Curva ROC - Conjunto TEST')
plt.legend()
plt.grid()
# Renderiza (dibuja) el gráfico en pantalla
plt.show()

print(f'ROC-AUC en TEST: {auc_test:.4f}')


## 13. Resumen compacto de métricas finales

Este bloque reúne las principales métricas para facilitar la comparación posterior con EfficientNetB0, ResNet152V2 u otros modelos.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_test = accuracy_score(y_test, y_pred)
precision_test = precision_score(y_test, y_pred, zero_division=0)
recall_test = recall_score(y_test, y_pred, zero_division=0)
f1_test = f1_score(y_test, y_pred, zero_division=0)
# Calcula el AUC (área bajo la curva ROC): una métrica de 0 a 1 de qué tan bien separa el modelo las dos clases
auc_test = roc_auc_score(y_test, y_prob)

print('=' * 45)
print('  MobileNetV3Small - RESULTADOS EN TEST')
print('=' * 45)
print(f'Accuracy : {accuracy_test:.4f}')
print(f'Precision: {precision_test:.4f}')
print(f'Recall   : {recall_test:.4f}')
print(f'F1-score : {f1_test:.4f}')
print(f'ROC-AUC  : {auc_test:.4f}')
print('=' * 45)


## 14. Guardar modelo final


In [ ]:
# Guarda el modelo entrenado (su arquitectura y pesos aprendidos) en un archivo .keras para reutilizarlo sin re-entrenar
model.save('MobileNetV3Small_final.keras')
print('Modelo guardado como MobileNetV3Small_final.keras')
